## Algorithms for Massive Dataset - project submission notebook

- Author: Andrea Colombo

### Project description

This notebook contains the implementation of the pagerank algorithm in python

### Step 1 - Environment setup

Here we simply pull in the required dependencies for the project, colab offers a wide variety of them but we need a couple extra ones to make this work
Some of these may be already included in the colab runtime, we pip install them just to be safe and avoid errors in the future

In [1]:
%pip install pyspark

In [3]:
import os

# This project needs a kaggle key in order to download the dataset, please set
# your own username and key in the lines below.
# NOTE: Do not commit the modified code anywhere public or your key will be
# leaked
os.environ['KAGGLE_USERNAME'] = 'x'
os.environ['KAGGLE_KEY'] = 'x'

if os.environ['KAGGLE_USERNAME'] == 'x' or os.environ['KAGGLE_KEY'] == 'x':
  print('!!!!!!! WARNING !!!!!!!')
  print('Provide values for KAGGLE_KEY and KAGGLE_USERNAME')
  print('!!!!!!! WARNING !!!!!!!')

!!!!!!! WARNING !!!!!!!
Provide values for KAGGLE_KEY and KAGGLE_USERNAME
!!!!!!! WARNING !!!!!!!


In [4]:
!kaggle datasets download -d benjaminawd/new-york-times-articles-comments-2020
!unzip new-york-times-articles-comments-2020.zip

Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
100% 1.95G/1.95G [00:19<00:00, 108MB/s]

Archive:  new-york-times-articles-comments-2020.zip
  inflating: nyt-articles-2020.csv   
  inflating: nyt-comments-2020.csv   
  inflating: nyt-comments-part0.csv  
  inflating: nyt-comments-part1.csv  
  inflating: nyt-comments-part2.csv  
  inflating: nyt-comments-part3.csv  
  inflating: nyt-comments-part4.csv  
  inflating: nyt-comments-part5.csv  
  inflating: nyt-comments-part6.csv  
  inflating: nyt-comments-part7.csv  
  inflating: nyt-comments-part8.csv  
  inflating: nyt-comments-part9.csv  
  inflating: test.csv                
  inflating: train.csv               


## Step 2 - Retrieve project dataset

The project dataset is open and retrievable from the web, we use

## Step 3 - Setup spark context and dataset retrieval



In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel

spark = (SparkSession.builder.appName('AMD-PROJECT').getOrCreate())
sc = spark.sparkContext
sc.setLogLevel('WARN')

print(f'Loaded spark with version {spark.version}')

Loaded spark with version 4.0.4


In [14]:
# The initial data frames are computed using the basic csv parsing provided by
# pyspark, we are going to filter these later to prepare them for the pagerank
# implementation

articles_df = (spark
               .read
               .option('header', True)
               .option('inferSchema', True)
               .option('mode', 'PERMISSIVE')
               .csv('nyt-articles-2020.csv'))

comments_df = (spark
               .read
               .option('header', True)
               .option('inferSchema', True)
               .csv('nyt-comments-2020.csv'))


print('===== Articles data frame snippet =====')
print('columns ', articles_df.columns)
articles_df.show(10, truncate=True)

print('===== Comments data frame snippet =====')
print('columns ', comments_df.columns)
comments_df.show(10, truncate=True)

===== Articles data frame snippet =====
columns  ['newsdesk', 'section', 'subsection', 'material', 'headline', 'abstract', 'keywords', 'word_count', 'pub_date', 'n_comments', 'uniqueID']
+---------+------------------+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+
| newsdesk|           section| subsection|           material|            headline|            abstract|            keywords|          word_count|            pub_date|     n_comments|            uniqueID|
+---------+------------------+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+
|Editorial|           Opinion|       NULL|          Editorial|Protect Veterans ...|Congress could do...|['Veterans', 'For...|                 680|2020-01-01 00:18:...|            186|nyt://article/6

In [24]:
# We can discard a lot of fields for this coputation, we only need the article
# id and title from the articles dataframe and we can discard the rest
# We also apply some basic duplicate cleaning and drop the null lines (although
# there shouldn't be any, we just do it to be sure we do not have runtime halts)

articles_cleaned = (articles_df
                    .select(
                        F.col('uniqueID').cast('string').alias('article_id'),
                        # The articles in this dataset do not have a proper
                        # title attribute so we use the headline instead
                        F.col('headline').cast('string').alias('article_title')
                    )
                    .filter(F.col('article_id').isNotNull())
                    # This check was added because some entries in the dataset
                    # seem to have 0 for ID (which doesn't really make sense)
                    # Snippet from tests:
                    # +--------------------+---------+
                    # |          article_id|  user_id|
                    # +--------------------+---------+
                    # |nyt://article/8bf...| 29778312|
                    # |nyt://article/8bf...| 10664125|
                    # |                   0| 65384034| <==== ????????????
                    # |nyt://article/176...| 75863404|
                    # |nyt://article/176...| 13800483|
                    # |nyt://article/176...| 59016607|
                    # |nyt://article/b31...| 69319909|
                    # |nyt://article/d56...| 79139306|
                    # |nyt://article/c95...| 58973206|
                    # |nyt://article/070...|102239845|
                    # +--------------------+---------+
                    #
                    # NOTE: with further testing I discovered that the article
                    # with the ID equal to 0 is the following (retrived using
                    # articles_cleaned.filter(F.col('article_id')=='0').show())
                    # +----------+--------------------+
                    # |article_id|       article_title|
                    # +----------+--------------------+
                    # |         0|Ben & Jerry’s Rad...|
                    # +----------+--------------------+
                    # Let's keep this check since this might be an issue with
                    # the dataset that may carry over with other versions
                    .filter(F.col('article_id') != '0')
                    .dropDuplicates(['article_id']))

comments_cleaned = (comments_df
                    .select(
                        F.col('articleID').cast('string').alias('article_id'),
                        F.col('userID').cast('string').alias('user_id')
                    )
                    .filter(F.col('user_id').isNotNull())
                    .filter(F.col('article_id').isNotNull())
                    .dropDuplicates(['article_id', 'user_id']))


articles_cleaned.printSchema()
comments_cleaned.printSchema()

root
 |-- article_id: string (nullable = true)
 |-- article_title: string (nullable = true)

root
 |-- article_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [25]:
joined_df = (comments_cleaned
             .join(
                 articles_cleaned.select('article_id'),
                 on='article_id',
                 how='inner'
             )
             .persist(StorageLevel.MEMORY_AND_DISK))

joined_df.printSchema()
joined_df.show(10, truncate=True)

# Now we construct the nodes of the graph we are going to operate on.
# In our graph, two articles (nodes) are connected by an edge if they
# share at least two users that commented.
# Two is a completely arbitrary threshold and it can be changed depending
# on the use case
#
# Graph -> G(V, E)
# given two arbitrary nodes i and j, with i != j, we have that:
# V(i,j) exists iff |CommentUserIDs(i) <intersect> CommentUserIDs(j)| >= 2
# V(i,j) exists => V(j,i) exists (undirected graph)
graph_nodes_df = (joined_df
                  .select('article_id')
                  .distinct()
                  .persist(StorageLevel.MEMORY_AND_DISK))

print("Articles in the graph (nodes) = ", graph_nodes_df.count())
print("Unique Article-User pairs = ", joined_df.count())

root
 |-- article_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+--------------------+---------+
|          article_id|  user_id|
+--------------------+---------+
|nyt://article/8bf...| 29778312|
|nyt://article/8bf...| 10664125|
|nyt://article/176...| 75863404|
|nyt://article/176...| 13800483|
|nyt://article/176...| 59016607|
|nyt://article/b31...| 69319909|
|nyt://article/d56...| 79139306|
|nyt://article/c95...| 58973206|
|nyt://article/070...|102239845|
|nyt://article/311...| 57822353|
+--------------------+---------+
only showing top 10 rows
Articles in the graph (nodes) =  15988
Unique Article-User pairs =  2087420
